In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
excel_path = "Dataset for Data Analytics.xlsx"
df = pd.read_excel(excel_path)

KeyError: 'TotalPrice_num'

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   OrderID          1200 non-null   str           
 1   Date             1200 non-null   datetime64[us]
 2   CustomerID       1200 non-null   str           
 3   Product          1200 non-null   str           
 4   Quantity         1200 non-null   int64         
 5   UnitPrice        1200 non-null   float64       
 6   ShippingAddress  1200 non-null   str           
 7   PaymentMethod    1200 non-null   str           
 8   OrderStatus      1200 non-null   str           
 9   TrackingNumber   1200 non-null   str           
 10  ItemsInCart      1200 non-null   int64         
 11  CouponCode       891 non-null    str           
 12  ReferralSource   1200 non-null   str           
 13  TotalPrice       1200 non-null   float64       
dtypes: datetime64[us](1), float64(2), int64(2), str(9)


In [4]:
df.isnull().sum()

OrderID              0
Date                 0
CustomerID           0
Product              0
Quantity             0
UnitPrice            0
ShippingAddress      0
PaymentMethod        0
OrderStatus          0
TrackingNumber       0
ItemsInCart          0
CouponCode         309
ReferralSource       0
TotalPrice           0
dtype: int64

In [5]:
df.duplicated()

0       False
1       False
2       False
3       False
4       False
        ...  
1195    False
1196    False
1197    False
1198    False
1199    False
Length: 1200, dtype: bool

In [6]:
df['OrderID'].duplicated().sum()

np.int64(0)

In [7]:
df.dtypes

OrderID                       str
Date               datetime64[us]
CustomerID                    str
Product                       str
Quantity                    int64
UnitPrice                 float64
ShippingAddress               str
PaymentMethod                 str
OrderStatus                   str
TrackingNumber                str
ItemsInCart                 int64
CouponCode                    str
ReferralSource                str
TotalPrice                float64
dtype: object

In [8]:
df.head()

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


In [9]:
df['CalculatedTotalPrice'] = (df['Quantity'] * df['UnitPrice']).round(2)

In [10]:
df['PriceDifference'] = (df['TotalPrice'].round(2) - df['CalculatedTotalPrice']).round(2)
mismatches = df[df['TotalPrice'].round(2) != df['CalculatedTotalPrice']]
print(f"Number of price mismatches found: {len(mismatches)}")

Number of price mismatches found: 0


In [11]:
#Imputation on missing values
df['CouponCode'] = df['CouponCode'].fillna('None')

In [12]:
print("--- Missing Values After Imputation ---")
print(df.isnull().sum())

--- Missing Values After Imputation ---
OrderID                 0
Date                    0
CustomerID              0
Product                 0
Quantity                0
UnitPrice               0
ShippingAddress         0
PaymentMethod           0
OrderStatus             0
TrackingNumber          0
ItemsInCart             0
CouponCode              0
ReferralSource          0
TotalPrice              0
CalculatedTotalPrice    0
PriceDifference         0
dtype: int64


In [13]:
print("\n--- Coupon Code Breakdown ---")
print(df['CouponCode'].value_counts())


--- Coupon Code Breakdown ---
CouponCode
FREESHIP    313
None        309
WINTER15    292
SAVE10      286
Name: count, dtype: int64


In [14]:
#Standardizing Date column to clean YYYY-MM-DD string format
df['Date'] = pd.to_datetime(df['Date']).dt.strftime('%Y-%m-%d')

In [15]:
# FINAL VERIFICATION:

# Verify Unique Identifier Integrity
order_duplicates = df['OrderID'].duplicated().sum()
tracking_duplicates = df['TrackingNumber'].duplicated().sum()

# Verify Date Format Integrity
invalid_dates = pd.to_datetime(df['Date'], errors='coerce').isnull().sum()

# Verify Null Values Count
remaining_nulls = df.isnull().sum().sum()

print(f"Duplicate OrderIDs:         {order_duplicates} (Target: 0)")
print(f"Duplicate Tracking Numbers: {tracking_duplicates} (Target: 0)")
print(f"Invalid / Unparsed Dates:   {invalid_dates} (Target: 0)")
print(f"Total Missing / Null Cells: {remaining_nulls} (Target: 0)")

if order_duplicates == 0 and tracking_duplicates == 0 and invalid_dates == 0 and remaining_nulls == 0:
    print("STATUS: PASSED - Dataset is verified as Gold Standard!")
else:
    print("STATUS: FAILED - Re-examine dataset cleaning steps.")

Duplicate OrderIDs:         0 (Target: 0)
Duplicate Tracking Numbers: 0 (Target: 0)
Invalid / Unparsed Dates:   0 (Target: 0)
Total Missing / Null Cells: 0 (Target: 0)
STATUS: PASSED - Dataset is verified as Gold Standard!


In [19]:
# Calculate all project metrics in one dictionary
metrics = {
    "Total Orders": len(df),
    "Total Revenue": f"${df['TotalPrice'].sum():,.2f}",
    "Average Order Value (AOV)": f"${df['TotalPrice'].mean():,.2f}",
    "Unique Customers": df['CustomerID'].nunique(),
    "Top Product Category": df.groupby('Product')['TotalPrice'].sum().idxmax(),
    "Top Product Revenue": f"${df.groupby('Product')['TotalPrice'].sum().max():,.2f}",
    "Start Date": df['Date'].min(),
    "End Date": df['Date'].max(),
    "Missing Values Count": df.isnull().sum().sum()
}

# Print all summary metrics
for metric, value in metrics.items():
    print(f"{metric}: {value}")

Total Orders: 1200
Total Revenue: $1,264,761.96
Average Order Value (AOV): $1,053.97
Unique Customers: 1189
Top Product Category: Chair
Top Product Revenue: $195,620.11
Start Date: 2023-01-01
End Date: 2025-06-30
Missing Values Count: 0
